In [ ]:
import os
import re
import glob
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import numpy as np
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

In [ ]:
MODEL = "gpt-4o-mini"

In [ ]:
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
openai = OpenAI()

In [ ]:
def clean_text(text):
    # Remove excessive spaces between letters
    text = re.sub(r'(?<=\w) (?=\w)', '', text)
    
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    
    # Optional: restore paragraph breaks if needed
    text = text.replace('. ', '.\n\n')
    
    return text.strip()

In [ ]:
def curatingDoc(data_folder):
    if not os.path.isdir(data_folder):
        raise ValueError(f"Folder does not exist: {data_folder}")
    
    documents = []
    
    pdf_files = [f for f in os.listdir(data_folder) if f.lower().endswith(".pdf")]
    for pdf_file in pdf_files:
        loader = PyPDFLoader(os.path.join(data_folder, pdf_file))
        for doc in loader.lazy_load():  # faster than load()
            doc.metadata["doc_type"] = "Data"
            doc.metadata["source"] = pdf_file
            doc.page_content = clean_text(doc.page_content)  # clean PDF text
            documents.append(doc)
    
    # --- Load TXT files ---
    txt_files = [f for f in os.listdir(data_folder) if f.lower().endswith(".txt")]
    for txt_file in txt_files:
        file_path = os.path.join(data_folder, txt_file)
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        text = clean_text(text)  # clean TXT text
        doc = Document(page_content=text, metadata={"doc_type": "Data", "source": txt_file})
        documents.append(doc)
    
    # --- Split into chunks ---
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=300)
    chunks = text_splitter.split_documents(documents)
    
    return chunks

In [ ]:
link = "../Data"
chunks = curatingDoc(link)
db_name = "vector_db"

In [ ]:
chunks

In [ ]:
#create embeddings
embeddings = OpenAIEmbeddings()

#create a vector store 
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

#create a vector store collection
collection = vectorstore._collection
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

In [ ]:
# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG; k is how many chunks to use
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [ ]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)